# 02 - pandas ile Veri Analizi ve Veri Ön İşleme
**Veri Madenciliği Dersi – Gümüşhane Üniversitesi, Yönetim Bilişim Sistemleri**

Veri madenciliği projelerinde zamanın büyük kısmı (%60–80) verinin **okunması, temizlenmesi ve dönüştürülmesi** ile geçer. pandas bu işlerin ana aracıdır.

### Öğrenme Hedefleri
1. `Series` ve `DataFrame` yapılarını tanımak
2. Veriyi okumak, ilk keşifsel incelemeyi yapmak (`head`, `info`, `describe`)
3. `loc` / `iloc` ve koşullu filtreleme ile veri seçmek
4. **Eksik veri, tekrar eden kayıt ve hatalı tip** sorunlarını çözmek
5. Yeni sütun türetmek, tarihlerle çalışmak
6. `groupby`, `pivot_table` ve `merge` ile özetleme ve birleştirme

> Bu defterde kurgusal bir **e-ticaret satış veri seti** kullanacağız. Veri, defterin içinde üretildiği için ek dosyaya ihtiyaç yoktur.

**Kullanım:** `✍️ Alıştırma` hücrelerini kendiniz doldurun. Çözümler en sondadır.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
print("pandas sürümü:", pd.__version__)

## 1. Series: Etiketli Tek Boyutlu Veri
`Series`, NumPy dizisine **indeks (etiket)** eklenmiş halidir.

In [ ]:
nufus = pd.Series(
    {"İstanbul": 15.7, "Ankara": 5.8, "İzmir": 4.5, "Trabzon": 0.82, "Gümüşhane": 0.14},
    name="Nüfus (milyon)"
)
print(nufus)
print("\nAnkara:", nufus["Ankara"])
print("1 milyondan büyük iller:\n", nufus[nufus > 1])

In [ ]:
print("Ortalama:", nufus.mean().round(2))
print("Sıralı:\n", nufus.sort_values(ascending=False))

## 2. DataFrame Oluşturma
`DataFrame` bir Excel tablosu gibidir: satırlar **gözlemler**, sütunlar **değişkenler (özellikler)**. Her sütun bir `Series`'tir.

In [ ]:
ogrenciler = pd.DataFrame({
    "ad":      ["Ayşe", "Mehmet", "Zeynep", "Emre", "Elif"],
    "bolum":   ["YBS", "İşletme", "YBS", "İktisat", "YBS"],
    "vize":    [70, 55, 92, 65, 80],
    "final":   [85, 48, 95, 72, 60],
})
ogrenciler

### Veri setimiz: E-ticaret siparişleri
Aşağıdaki hücre, bilerek **kirli** (eksik değerli, tekrar eden, hatalı) bir veri seti üretir. Gerçek hayatta veriyi genellikle `pd.read_csv("dosya.csv")` veya `pd.read_excel("dosya.xlsx")` ile okursunuz.

In [ ]:
rng = np.random.default_rng(42)
n = 300

urunler = {
    "Laptop": ("Elektronik", 32000), "Kulaklık": ("Elektronik", 1500),
    "Telefon": ("Elektronik", 25000), "Mont": ("Giyim", 3200),
    "Ayakkabı": ("Giyim", 2400), "Tişört": ("Giyim", 450),
    "Kitap": ("Kitap", 220), "Defter": ("Kırtasiye", 60),
    "Kalem Seti": ("Kırtasiye", 180), "Kahve Makinesi": ("Ev", 4800),
}
iller = ["İstanbul", "Ankara", "İzmir", "Trabzon", "Gümüşhane", "Erzurum", "Bursa"]

urun_sec = rng.choice(list(urunler), size=n)
df = pd.DataFrame({
    "siparis_id": np.arange(1001, 1001 + n),
    "tarih": pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, n), unit="D"),
    "musteri_id": rng.integers(1, 81, n),
    "il": rng.choice(iller, size=n, p=[0.3, 0.2, 0.15, 0.1, 0.05, 0.1, 0.1]),
    "urun": urun_sec,
    "kategori": [urunler[u][0] for u in urun_sec],
    "birim_fiyat": [urunler[u][1] * rng.uniform(0.9, 1.1) for u in urun_sec],
    "adet": rng.integers(1, 5, n),
    "odeme": rng.choice(["Kredi Kartı", "Havale", "Kapıda Ödeme"], size=n, p=[0.65, 0.2, 0.15]),
    "puan": rng.choice([1, 2, 3, 4, 5], size=n, p=[0.05, 0.08, 0.17, 0.35, 0.35]).astype(float),
})
df["birim_fiyat"] = df["birim_fiyat"].round(2)

# --- Veriyi bilerek kirletiyoruz ---
df.loc[rng.choice(n, 25, replace=False), "puan"] = np.nan          # eksik puanlar
df.loc[rng.choice(n, 8, replace=False), "il"] = np.nan             # eksik iller
df.loc[rng.choice(n, 3, replace=False), "adet"] = -1               # hatalı adet
df.loc[[5, 17], "il"] = ["  ankara ", "İSTANBUL"]                  # yazım tutarsızlığı
df = pd.concat([df, df.iloc[[10, 20, 30, 40]]], ignore_index=True) # tekrar eden satırlar

print("Veri seti hazır:", df.shape)

In [ ]:
df

## 3. İlk Keşif (Keşifsel Veri Analizinin İlk Adımı) ⭐
Yeni bir veri setiyle karşılaştığınızda **her zaman** şu komutlarla başlayın:

In [ ]:
df.head()        # ilk 5 satır (tail() son satırlar, sample() rastgele satırlar)

In [ ]:
df.info()        # sütun tipleri, eksik olmayan değer sayısı, bellek

In [ ]:
df.describe()    # sayısal sütunların özet istatistikleri

In [ ]:
df.describe(exclude="number")   # sayısal olmayan (metin, tarih) sütunların özeti

In [ ]:
print("Şekil:", df.shape)
print("Sütunlar:", list(df.columns))
print("\nKategori dağılımı:")
print(df["kategori"].value_counts())
print("\nÖdeme yöntemi oranları:")
print(df["odeme"].value_counts(normalize=True).round(3))
print("\nBenzersiz il sayısı:", df["il"].nunique())

> 🔎 **Gözlem soruları:** `describe()` çıktısında `adet` sütununun minimum değerine bakın — mantıklı mı? `info()` çıktısında hangi sütunlarda eksik değer var? Bu soruları 6. bölümde çözeceğiz.

## 4. Sütun ve Satır Seçimi: `[]`, `loc`, `iloc` ⭐
- `df["sütun"]` → tek sütun (Series)  |  `df[["s1", "s2"]]` → birden çok sütun (DataFrame)
- `loc` → **etiket** ile seçim: `df.loc[satır_etiketi, sütun_adı]` (dilimde bitiş **dahil**)
- `iloc` → **konum** ile seçim: `df.iloc[satır_no, sütun_no]` (dilimde bitiş **dahil değil**)

In [ ]:
df[["urun", "birim_fiyat", "adet"]].head(3)

In [ ]:
print(df.loc[0:3, ["urun", "il"]])      # 0,1,2,3 -> 4 satır
print()
print(df.iloc[0:3, 0:4])                # 0,1,2 -> 3 satır, ilk 4 sütun

## 5. Koşullu Filtreleme
NumPy'daki boolean maskeleme mantığı aynen geçerlidir: `&`, `|`, `~` ve **parantezler**.

In [ ]:
# Trabzon'dan verilen elektronik siparişler
df[(df["il"] == "Trabzon") & (df["kategori"] == "Elektronik")].head()

In [ ]:
# isin: birden fazla değerden biri
karadeniz = df[df["il"].isin(["Trabzon", "Gümüşhane"])]
print("Karadeniz siparişleri:", len(karadeniz))

# query: daha okunabilir alternatif
df.query("birim_fiyat > 20000 and adet >= 3").head()

In [ ]:
# Metin işlemleri: .str erişimcisi
df[df["urun"].str.contains("Kahve")].head(3)

### ✍️ Alıştırma 1
1. Sadece `urun`, `kategori` ve `puan` sütunlarını içeren ilk 5 satırı gösterin.
2. Kapıda ödeme ile yapılmış ve puanı 2 veya altında olan siparişlerin sayısını bulun.
3. Giyim veya Kitap kategorisinde olup birim fiyatı 1000 TL'den düşük siparişleri listeleyin.

In [ ]:
# Kodunuzu buraya yazın

## 6. Veri Temizleme ⭐⭐
Veri madenciliğinde **"çöp girerse çöp çıkar" (garbage in, garbage out)** kuralı geçerlidir. Modelleme öncesi bu adımlar mutlaka yapılmalıdır.

### 6.1 Tekrar eden kayıtlar

In [ ]:
print("Tekrar eden satır sayısı:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Temizlik sonrası şekil:", df.shape)

### 6.2 Tutarsız metin değerleri

In [ ]:
print("Önce:", df["il"].unique())

df["il"] = df["il"].str.strip().str.title()
df["il"] = df["il"].replace({"Istanbul": "İstanbul"})   # Python'un title() fonksiyonu Türkçe İ harfini bilmez!

print("Sonra:", df["il"].unique())

> ⚠️ **Türkçe karakter tuzağı:** `"İSTANBUL".lower()` sonucu `"i̇stanbul"` olur (noktalı i + birleşik nokta). Türkçe metinlerde büyük/küçük harf dönüşümlerinden sonra sonucu mutlaka kontrol edin.

### 6.3 Mantıksız (hatalı) değerler

In [ ]:
print("Hatalı adet içeren satırlar:")
print(df[df["adet"] <= 0][["siparis_id", "urun", "adet"]])

df = df[df["adet"] > 0].copy()
print("\nYeni şekil:", df.shape)

### 6.4 Eksik değerler
Üç temel strateji vardır: **silmek**, **sabit bir değerle doldurmak**, **istatistiksel bir değerle (ortalama/medyan/mod) doldurmak**. Seçim, verinin anlamına bağlıdır.

In [ ]:
print("Sütun bazında eksik değer sayısı:")
print(df.isna().sum())
print("\nEksik oranı (%):")
print((df.isna().mean() * 100).round(2))

In [ ]:
# il: kategorik değişken -> en sık görülen değer (mod) ya da "Bilinmiyor"
df["il"] = df["il"].fillna("Bilinmiyor")

# puan: sayısal değişken -> ürünün kendi medyan puanı ile doldur (daha akıllı yöntem)
df["puan"] = df["puan"].fillna(df.groupby("urun")["puan"].transform("median"))

print(df.isna().sum().sum(), "eksik değer kaldı")

> 💡 `dropna()` ile eksik satırları silmek en kolay yoldur ama veri kaybına yol açar. Eksik oranı çok düşükse (%1–2) silmek kabul edilebilir.

## 7. Yeni Sütun Türetme (Feature Engineering)

In [ ]:
df["tutar"] = (df["birim_fiyat"] * df["adet"]).round(2)

# Tarihten bilgi çıkarma: .dt erişimcisi
df["ay"] = df["tarih"].dt.month
df["gun_adi"] = df["tarih"].dt.day_name()
df["hafta_sonu"] = df["tarih"].dt.dayofweek >= 5

# Koşullu sütun
df["kargo"] = np.where(df["tutar"] >= 1000, "Ücretsiz", "Ücretli")

df[["tarih", "ay", "gun_adi", "hafta_sonu", "tutar", "kargo"]].head()

In [ ]:
# pd.cut: sayısal değişkeni aralıklara bölme (ayrıklaştırma / binning)
df["tutar_grubu"] = pd.cut(
    df["tutar"],
    bins=[0, 500, 5000, 50000, np.inf],
    labels=["Düşük", "Orta", "Yüksek", "Çok Yüksek"],
)
df["tutar_grubu"].value_counts()

In [ ]:
# map ve apply ile dönüşüm
mevsim = {12: "Kış", 1: "Kış", 2: "Kış", 3: "İlkbahar", 4: "İlkbahar", 5: "İlkbahar",
          6: "Yaz", 7: "Yaz", 8: "Yaz", 9: "Sonbahar", 10: "Sonbahar", 11: "Sonbahar"}
df["mevsim"] = df["ay"].map(mevsim)

def memnuniyet(puan):
    if puan >= 4:
        return "Memnun"
    elif puan == 3:
        return "Nötr"
    return "Memnun Değil"

df["memnuniyet"] = df["puan"].apply(memnuniyet)
df[["ay", "mevsim", "puan", "memnuniyet"]].sample(5, random_state=1)

### ✍️ Alıştırma 2
1. `df`'ye `kdv_dahil` adında, `tutar` değerinin %20 KDV eklenmiş halini içeren bir sütun ekleyin.
2. `ceyrek` adında, siparişin yılın kaçıncı çeyreğinde verildiğini gösteren bir sütun ekleyin. (İpucu: `.dt.quarter`)
3. Hafta sonu verilen siparişlerin toplam tutarını bulun.

In [ ]:
# Kodunuzu buraya yazın

## 8. Sıralama

In [ ]:
df.sort_values("tutar", ascending=False)[["siparis_id", "urun", "il", "tutar"]].head()

In [ ]:
# Birden fazla sütuna göre: önce kategori (A-Z), sonra tutar (büyükten küçüğe)
df.sort_values(["kategori", "tutar"], ascending=[True, False])[["kategori", "urun", "tutar"]].head(8)

## 9. Gruplama: `groupby` ⭐⭐
**Böl → Uygula → Birleştir (split-apply-combine)** mantığı. SQL'deki `GROUP BY` ile aynıdır.

In [ ]:
# Kategori bazında toplam ciro
df.groupby("kategori")["tutar"].sum().sort_values(ascending=False)

In [ ]:
# Birden fazla istatistik: agg
df.groupby("il").agg(
    siparis_sayisi=("siparis_id", "count"),
    toplam_ciro=("tutar", "sum"),
    ortalama_sepet=("tutar", "mean"),
    ortalama_puan=("puan", "mean"),
).round(1).sort_values("toplam_ciro", ascending=False)

In [ ]:
# Birden fazla sütuna göre gruplama
df.groupby(["kategori", "odeme"])["tutar"].mean().round(0).unstack()

## 10. Pivot Tablo ve Çapraz Tablo
Excel'deki Özet Tablo (Pivot Table) özelliğinin aynısıdır.

In [ ]:
pd.pivot_table(
    df, values="tutar", index="kategori", columns="mevsim",
    aggfunc="sum", fill_value=0, margins=True, margins_name="Toplam"
).round(0)

In [ ]:
# crosstab: iki kategorik değişkenin frekans tablosu
pd.crosstab(df["kategori"], df["memnuniyet"], normalize="index").round(2)

### ✍️ Alıştırma 3
1. Her ürün için satılan **toplam adet** ve **ortalama puan**ı tek tabloda gösterin; toplam adede göre büyükten küçüğe sıralayın.
2. Aylara göre toplam ciroyu hesaplayın. En yüksek ciro hangi ayda gerçekleşmiş?
3. Satırlarda `il`, sütunlarda `kategori` olan ve hücrelerde **sipariş sayısını** gösteren bir pivot tablo oluşturun.

In [ ]:
# Kodunuzu buraya yazın

## 11. Tabloları Birleştirme: `merge` ve `concat`
Gerçek projelerde veri tek tabloda gelmez. Örneğin müşteri bilgileri ayrı bir tabloda tutulur.

In [ ]:
rng2 = np.random.default_rng(1)
musteriler = pd.DataFrame({
    "musteri_id": np.arange(1, 76),     # dikkat: 76-80 arası müşteriler bu tabloda yok!
    "cinsiyet": rng2.choice(["K", "E"], size=75),
    "yas": rng2.integers(18, 65, size=75),
    "uyelik": rng2.choice(["Standart", "Premium"], size=75, p=[0.7, 0.3]),
})
musteriler.head()

In [ ]:
inner = df.merge(musteriler, on="musteri_id", how="inner")
left  = df.merge(musteriler, on="musteri_id", how="left")

print("Sipariş tablosu  :", df.shape)
print("inner join sonucu:", inner.shape, "-> eşleşmeyenler atıldı")
print("left join sonucu :", left.shape, "-> tüm siparişler korundu, eşleşmeyenlerde NaN")
print("Eşleşmeyen sipariş:", left["uyelik"].isna().sum())

In [ ]:
# Birleştirilmiş veriyle analiz: üyelik tipine göre ortalama sepet tutarı
inner.groupby("uyelik")["tutar"].agg(["count", "mean", "median"]).round(0)

In [ ]:
# concat: aynı yapıdaki tabloları alt alta ekleme
ocak = df[df["ay"] == 1]
subat = df[df["ay"] == 2]
ilk_iki_ay = pd.concat([ocak, subat], ignore_index=True)
print(len(ocak), "+", len(subat), "=", len(ilk_iki_ay))

## 12. Zaman Serisi Özetleme ve Dosyaya Kaydetme

In [ ]:
aylik = df.set_index("tarih").resample("ME")["tutar"].sum()   # ME = ay sonu
aylik.index = aylik.index.strftime("%Y-%m")
aylik.round(0)

In [ ]:
# Temizlenmiş veriyi kaydetme (sonraki defterlerde kullanabilirsiniz)
df.to_csv("eticaret_temiz.csv", index=False, encoding="utf-8-sig")   # utf-8-sig: Excel'de Türkçe karakterler düzgün görünür
kontrol = pd.read_csv("eticaret_temiz.csv", parse_dates=["tarih"])
print("Kaydedildi ve tekrar okundu:", kontrol.shape)

### ✍️ Alıştırma 4 (Kapsamlı): RFM Analizi
RFM, müşteri segmentasyonunda kullanılan klasik bir veri madenciliği yöntemidir:
- **R (Recency):** Müşterinin son alışverişinden bu yana geçen gün sayısı (analiz tarihi: `2026-01-01`)
- **F (Frequency):** Toplam sipariş sayısı
- **M (Monetary):** Toplam harcama tutarı

Görevler:
1. `df`'yi `musteri_id`'ye göre gruplayarak her müşteri için R, F, M değerlerini içeren `rfm` tablosunu oluşturun.
2. M değerine göre en değerli 5 müşteriyi listeleyin.
3. `pd.qcut` kullanarak M değerini 3 eşit gruba bölün (`"Bronz"`, `"Gümüş"`, `"Altın"`) ve her grupta kaç müşteri olduğunu gösterin.

In [ ]:
# Kodunuzu buraya yazın

---
## 📌 Özet
| Adım | Komutlar |
|---|---|
| Okuma / yazma | `read_csv`, `read_excel`, `to_csv` |
| İlk keşif | `head`, `info`, `describe`, `value_counts`, `nunique` |
| Seçim | `df[...]`, `loc`, `iloc`, `query`, `isin`, `.str.contains` |
| Temizlik | `duplicated`, `drop_duplicates`, `isna`, `fillna`, `dropna`, `.str.strip` |
| Türetme | aritmetik, `np.where`, `pd.cut`, `map`, `apply`, `.dt` |
| Özetleme | `groupby` + `agg`, `pivot_table`, `crosstab`, `resample` |
| Birleştirme | `merge (inner/left/right/outer)`, `concat` |

**Sonraki defter:** `03_matplotlib.ipynb` — veriyi görselleştirme.

---
## ✅ Çözümler

### Çözüm 1

In [ ]:
print(df[["urun", "kategori", "puan"]].head())
print("\nKapıda ödeme & puan <= 2:", len(df[(df["odeme"] == "Kapıda Ödeme") & (df["puan"] <= 2)]))
df[df["kategori"].isin(["Giyim", "Kitap"]) & (df["birim_fiyat"] < 1000)].head()

### Çözüm 2

In [ ]:
df["kdv_dahil"] = (df["tutar"] * 1.20).round(2)
df["ceyrek"] = df["tarih"].dt.quarter
print("Hafta sonu toplam tutar:", df.loc[df["hafta_sonu"], "tutar"].sum().round(2))
df[["tutar", "kdv_dahil", "tarih", "ceyrek"]].head()

### Çözüm 3

In [ ]:
print(df.groupby("urun").agg(toplam_adet=("adet", "sum"), ortalama_puan=("puan", "mean"))
        .round(2).sort_values("toplam_adet", ascending=False))

aylik_ciro = df.groupby("ay")["tutar"].sum()
print("\nEn yüksek cirolu ay:", aylik_ciro.idxmax(), "->", aylik_ciro.max().round(0), "TL")

pd.pivot_table(df, values="siparis_id", index="il", columns="kategori", aggfunc="count", fill_value=0)

### Çözüm 4

In [ ]:
analiz_tarihi = pd.Timestamp("2026-01-01")
rfm = df.groupby("musteri_id").agg(
    R=("tarih", lambda t: (analiz_tarihi - t.max()).days),
    F=("siparis_id", "count"),
    M=("tutar", "sum"),
)
print(rfm.head())
print("\nEn değerli 5 müşteri:")
print(rfm.sort_values("M", ascending=False).head().round(0))

rfm["segment"] = pd.qcut(rfm["M"], q=3, labels=["Bronz", "Gümüş", "Altın"])
print("\n", rfm["segment"].value_counts())